# RAG chain 구현

## 학습 목표

- PDF 문서를 로드하고 청킹, 임베딩, 벡터 DB 저장까지 RAG 준비 과정을 구현한다
- LCEL 방식의 RAG 체인을 구성하고 질의응답을 실행한다
- 출 처 표시 기능을 추가해 신뢰성 있는 답변 시스템을 만든다

---

## 전체 파이프라인 요약

```
문서 파일
  ↓ TextLoader / PyPDFLoader
Document 리스트
  ↓ RecursiveCharacterTextSplitter
청크(Chunk) 리스트
  ↓ OpenAIEmbeddings(text-embedding-3-small)
벡터
  ↓ Chroma.from_documents()
벡터 DB (로컬 저장)
  ↓ .as_retriever()
Retriever
  ↓ LCEL 체인 {context: retriever | format_docs, question: ...}
RAG 체인
  ↓ .invoke(질문)
답변
```

## 핵심 정리

| 단계 | 코드 | 설명 |
|---|---|---|
| 로드 | `TextLoader`, `PyPDFLoader` | 문서를 Document 형식으로 |
| 청킹 | `RecursiveCharacterTextSplitter` | 작은 단위로 분할 |
| 임베딩 | `OpenAIEmbeddings(model='text-embedding-3-small')` | 텍스트→벡터 |
| 저장 | `Chroma.from_documents(...)` | 벡터 DB 생성 |
| 검색 | `.as_retriever()` | 유사 청크 반환 |
| 체인 | `{context: retriever\|format_docs, ...}\|prompt\|llm\|parser` | LCEL RAG |

## 사전 설치 패키지
```
langchain langchain-openai langchain-community langchain-chroma
langchain-text-splitters pypdf python-dotenv
```

In [ ]:
# pip install langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters pypdf python-dotenv -q

## step 0. 환경 설정

In [1]:

import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv('OPENAI_API_KEY'), '❌ OPENAI_API_KEY를 설정해주세요.'
print('✅ 환경 설정 완료')



✅ 환경 설정 완료


## Step 1. 실습용 문서 준비

실습을 위해 텍스트 파일을 직접 만들어 사용합니다. 실제 서비스에서 PDF, 웹페이지, DB 등 다양한 형식을 사용합니다.

In [3]:
# 실습용 가상 회사 정책 문서 생성
sample_text = """
# ABC 회사 직원 복리후생 안내

## 연차 휴가
입사 1년 미만 직원은 월 1일씩 최대 11일의 연차를 받습니다.
입사 1년 이상 직원은 연 15일의 연차를 받습니다.
3년 이상 근무 시 매 2년마다 1일씩 추가됩니다. 최대 25일까지 부여됩니다.

## 재택근무
모든 정규직 직원은 주 2일 재택근무가 가능합니다.
재택근무 신청은 매주 금요일까지 HR 시스템에 등록해야 합니다.
신입사원(입사 6개월 미만)은 재택근무가 제한됩니다.

## 의료비 지원
연간 의료비 100만원까지 지원됩니다.
치과, 안과 포함 모든 의료비가 해당됩니다.
영수증 제출 후 15일 이내에 급여 계좌로 입금됩니다.

## 교육 지원
직무 관련 자격증 취득 시 응시료 전액을 지원합니다.
온라인 강의 플랫폼(인프런, Coursera 등) 연간 50만원까지 지원됩니다.
사내 도서 구매 비용은 월 3만원 한도로 지원됩니다.

## 경조사 지원
결혼 축하금: 50만원 + 5일 유급 휴가
출산 축하금: 100만원 + 남성 3일, 여성 90일 유급 육아휴직
부모님 장례: 5일 유급 휴가 + 위로금 30만원

## 식대 지원
점심 식대 1일 1만원 지원 (월 최대 22일)
야근 시 저녁 식대 1만 5천원 추가 지원
사내 카페테리아에서 아침 무료 제공 (오전 8시~9시)
"""

# 파일로 저장
with open('company_policy.txt', 'w', encoding='utf-8') as f:
    f.write(sample_text)

print('✅ 실습용 문서 생성 완료 (company_policy.txt)')
print(f'문서 길이: {len(sample_text)}자')

✅ 실습용 문서 생성 완료 (company_policy.txt)
문서 길이: 644자


## Step 2. 문서 로드

텍스트 파일을 LangChain Documnet 형식으로 로드합니다.

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('company_policy.txt', encoding='utf-8')
docs = loader.load()

print(f'로드된 문서 수: {len(docs)}')
print(f'문서 내용 미리보기 (첫 200자):')
print(docs[0].page_content[:200])
print(f'\n메타데이터: {docs[0].metadata}')

C:\Users\qkrru\AppData\Local\Temp\ipykernel_21936\1299602468.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


로드된 문서 수: 1
문서 내용 미리보기 (첫 200자):

# ABC 회사 직원 복리후생 안내

## 연차 휴가
입사 1년 미만 직원은 월 1일씩 최대 11일의 연차를 받습니다.
입사 1년 이상 직원은 연 15일의 연차를 받습니다.
3년 이상 근무 시 매 2년마다 1일씩 추가됩니다. 최대 25일까지 부여됩니다.

## 재택근무
모든 정규직 직원은 주 2일 재택근무가 가능합니다.
재택근무 신청은 매주 금요일까지 H

메타데이터: {'source': 'company_policy.txt'}


## Step 3. 청킹

긴 문서를 작은 단위로 나눕니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, # 청크 최대 300자
    chunk_overlap=30, # 앞뒤 30자씩 겹침 (문맥 보존)
    separators=["\n\n", "\n", " ", ""] # 이 순서로 분할
)

chunks = splitter.split_documents(docs)

print(f'전체 문서: {len(docs)}개 -> 청크: {len(chunks)}개')
print('\n--- 첫 번째 청크 ---')
print(chunks[0].page_content)
print('\n--- 두 번째 청크 ---')
print(chunks[1].page_content)
print('\n--- 세 번째 청크 ---')
print(chunks[2].page_content)


전체 문서: 1개 -> 청크: 3개

--- 첫 번째 청크 ---
# ABC 회사 직원 복리후생 안내

## 연차 휴가
입사 1년 미만 직원은 월 1일씩 최대 11일의 연차를 받습니다.
입사 1년 이상 직원은 연 15일의 연차를 받습니다.
3년 이상 근무 시 매 2년마다 1일씩 추가됩니다. 최대 25일까지 부여됩니다.

## 재택근무
모든 정규직 직원은 주 2일 재택근무가 가능합니다.
재택근무 신청은 매주 금요일까지 HR 시스템에 등록해야 합니다.
신입사원(입사 6개월 미만)은 재택근무가 제한됩니다.

--- 두 번째 청크 ---
## 의료비 지원
연간 의료비 100만원까지 지원됩니다.
치과, 안과 포함 모든 의료비가 해당됩니다.
영수증 제출 후 15일 이내에 급여 계좌로 입금됩니다.

## 교육 지원
직무 관련 자격증 취득 시 응시료 전액을 지원합니다.
온라인 강의 플랫폼(인프런, Coursera 등) 연간 50만원까지 지원됩니다.
사내 도서 구매 비용은 월 3만원 한도로 지원됩니다.

--- 세 번째 청크 ---
## 경조사 지원
결혼 축하금: 50만원 + 5일 유급 휴가
출산 축하금: 100만원 + 남성 3일, 여성 90일 유급 육아휴직
부모님 장례: 5일 유급 휴가 + 위로금 30만원

## 식대 지원
점심 식대 1일 1만원 지원 (월 최대 22일)
야근 시 저녁 식대 1만 5천원 추가 지원
사내 카페테리아에서 아침 무료 제공 (오전 8시~9시)


## Step 4. 임베딩 & 벡터 DB 저장

청크를 임베딩하고 Chroma DB에 저장합니다.

**이 단계에서 가장 시간이 오래걸린다 - OpenAI API를 호출해 모든 청크를 변환한다**

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 모델 초기화 
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# 벡터 DB 생성 및 저장
print('⏳ 임베딩 생성 중...')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_db' # 로컬 저장
)

print(f'벡터 DB 생성 완료')
print(f'저장된 청크 수: {vectorstore._collection.count()}')

⏳ 임베딩 생성 중...
벡터 DB 생성 완료
저장된 청크 수: 3


## Step 5. 검색 테스트

질문을 던졌을 때 어떤 청크가 검색되는지 확인합니다.

In [10]:
# 검색기 생성
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2} # 상위 2개 청크 반환
)

# 검색 테스트
query = '연차는 며칠이나 받을 수 있나요?'
results = retriever.invoke(query)

print(f'질문: {query}')
print(f'검색된 청크 수: {len(results)}')
for i, doc in enumerate(results, 1):
    print(f'\n[청크 {i}]')
    print(doc.page_content)

질문: 연차는 며칠이나 받을 수 있나요?
검색된 청크 수: 2

[청크 1]
# ABC 회사 직원 복리후생 안내

## 연차 휴가
입사 1년 미만 직원은 월 1일씩 최대 11일의 연차를 받습니다.
입사 1년 이상 직원은 연 15일의 연차를 받습니다.
3년 이상 근무 시 매 2년마다 1일씩 추가됩니다. 최대 25일까지 부여됩니다.

## 재택근무
모든 정규직 직원은 주 2일 재택근무가 가능합니다.
재택근무 신청은 매주 금요일까지 HR 시스템에 등록해야 합니다.
신입사원(입사 6개월 미만)은 재택근무가 제한됩니다.

[청크 2]
## 경조사 지원
결혼 축하금: 50만원 + 5일 유급 휴가
출산 축하금: 100만원 + 남성 3일, 여성 90일 유급 육아휴직
부모님 장례: 5일 유급 휴가 + 위로금 30만원

## 식대 지원
점심 식대 1일 1만원 지원 (월 최대 22일)
야근 시 저녁 식대 1만 5천원 추가 지원
사내 카페테리아에서 아침 무료 제공 (오전 8시~9시)


## Step 6. LCEL RAG 체인 구성

검색기 + 프롬프트 + LLM을 LCEL로 연결합니다.

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# RAG 프롬프트
rag_prompt = ChatPromptTemplate.from_template("""
당신은 ABC 회사의 HR 어시스턴트입니다.
아래 제공된 회사 정책 문서를 바탕으로 직원의 질문에 정확하게 답변해주세요.
문서에 없는 내용은 '해당 정보를 문서에서 찾을 수 없습니다'라고 답하세요.

[회사 정책 문서]
{context}

[직원 질문]
{question}

[답변]
""")

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# LECL RAG 체인
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print('✅ RAG 체인 구성 완료')

✅ RAG 체인 구성 완료


## Step 7. RAG 질의응답 테스트



In [12]:
# 여러 질문 테스트
questions = [
    '연차는 몇 일 받을 수 있나요?',
    '재택근무는 신입사원도 가능한가요?',
    '결혼하면 어떤 지원을 받을 수 있나요?',
    '의료비는 얼마까지 지원되나요?',
    '회사 주차장은 몇 대까지 주차 가능한가요?'  # 문서에 없는 정보
]

for question in questions:
    print(f'\n{'='*60}')
    print(f'❓ 질문: {question}')
    print(f'💬 답변: {rag_chain.invoke(question)}')


❓ 질문: 연차는 몇 일 받을 수 있나요?
💬 답변: 입사 1년 미만 직원은 월 1일씩 최대 11일의 연차를 받습니다. 입사 1년 이상 직원은 연 15일의 연차를 받습니다. 3년 이상 근무 시 매 2년마다 1일씩 추가되어 최대 25일까지 부여됩니다.

❓ 질문: 재택근무는 신입사원도 가능한가요?
💬 답변: 신입사원(입사 6개월 미만)은 재택근무가 제한됩니다.

❓ 질문: 결혼하면 어떤 지원을 받을 수 있나요?
💬 답변: 결혼 시에는 50만원의 축하금과 5일의 유급 휴가를 지원받을 수 있습니다.

❓ 질문: 의료비는 얼마까지 지원되나요?
💬 답변: 연간 의료비 100만원까지 지원됩니다.

❓ 질문: 회사 주차장은 몇 대까지 주차 가능한가요?
💬 답변: 해당 정보를 문서에서 찾을 수 없습니다.


## Stemp 8. 출처 표시 RAG (신뢰성 향상)

답변과 함께 어떤 문서 청크에서 가져왔는지 표시합니다.

In [13]:
from langchain_core.runnables import RunnableParallel

# 컨텍스트와 답변을 함께 반환하는 체인
rag_chain_with_source = RunnableParallel(
    context=retriever,
    question=RunnablePassthrough()
).assign(
    answer=(
        lambda x: {'context': format_docs(x['context']), 'question': x['question']}
    ) | rag_prompt | llm | StrOutputParser()
)

# 실행
result = rag_chain_with_source.invoke('교육비 지원은 어떻게 되나요?')

print('💬 답변:')
print(result['answer'])
print('\n📚 참고한 문서 청크:')
for i, doc in enumerate(result['context'], 1):
    print(f'\n[출처 {i}]')
    print(doc.page_content[:150] + '...')


💬 답변:
직무 관련 자격증 취득 시 응시료 전액을 지원합니다. 또한, 온라인 강의 플랫폼(인프런, Coursera 등)에서 연간 50만원까지 지원되며, 사내 도서 구매 비용은 월 3만원 한도로 지원됩니다.

📚 참고한 문서 청크:

[출처 1]
## 의료비 지원
연간 의료비 100만원까지 지원됩니다.
치과, 안과 포함 모든 의료비가 해당됩니다.
영수증 제출 후 15일 이내에 급여 계좌로 입금됩니다.

## 교육 지원
직무 관련 자격증 취득 시 응시료 전액을 지원합니다.
온라인 강의 플랫폼(인프런, Course...

[출처 2]
## 경조사 지원
결혼 축하금: 50만원 + 5일 유급 휴가
출산 축하금: 100만원 + 남성 3일, 여성 90일 유급 육아휴직
부모님 장례: 5일 유급 휴가 + 위로금 30만원

## 식대 지원
점심 식대 1일 1만원 지원 (월 최대 22일)
야근 시 저녁 식대 1만...


## Step 9. 기존 DB 불러오기

한 번 만든 벡터 DB는 저장해두고 다시 불러올 수 있습니다.


In [14]:
# 저장된 DB 불러오기 (새 세션에서도 사용 가능)
loaded_vectorstore = Chroma(
    persist_directory='./chroma_db',
    embedding_function=embeddings
)

print(f'✅ DB 불러오기 완료 (청크 수: {loaded_vectorstore._collection.count()})')

# 불러온 DB로 검색 테스트
loaded_retriever = loaded_vectorstore.as_retriever(search_kwargs={'k': 2})
results = loaded_retriever.invoke('식대 지원')
print(f'검색 결과: {results[0].page_content[:100]}...')


✅ DB 불러오기 완료 (청크 수: 3)
검색 결과: ## 경조사 지원
결혼 축하금: 50만원 + 5일 유급 휴가
출산 축하금: 100만원 + 남성 3일, 여성 90일 유급 육아휴직
부모님 장례: 5일 유급 휴가 + 위로금 30만원
...
